# LLM4IAS: 22 test gốc × Qwen2-7B-Instruct × rubric 2409 v1

Không phải dataset SOP aircraft. Chạy 22 Inspection Station cases từ workbook
bài 2409, cùng checkpoint và settings Qwen2 trước đây. Rubric không đưa vào
inference prompt; thay cách chấm không được làm lộ đáp án cho model.

Rubric v1 do người dùng chấp nhận: F (JSON), C (đúng command về nội dung),
R (reason 1–5). F và strict AST tính tự động. C_suggested chỉ là gợi ý;
thông báo alert khác câu chữ, raw lỗi JSON và case reference conflict giữ U.
C_final và R_final cần người duyệt, không giả lập nhãn con người bằng regex.

Chấm lại cả raw output của model mới và raw Qwen2 tác giả trong cùng sheet
Qwen2-7B-Instruct_1. Không sửa lỗi JSON của tác giả, không dùng G/H thay rubric mới.
Có 44 response A/B để duyệt; không mở source_key/auto_suggestions trước khi chấm
nếu muốn che nguồn. Che nguồn không bảo đảm reviewer không đoán được phong cách.

Kaggle session mới, Internet ON, ưu tiên T4 x2. Không cần API key. Giữ FP16,
greedy, 256 output token/case như run 22 test trước. Không dùng protocol patch
của SOP aircraft. Đổi rubric không nhất thiết phải infer lại nếu còn raw run cũ.


## 1. Cài thư viện
Giữ PyTorch/CUDA có sẵn của Kaggle. Các phiên bản dưới đây do mình chọn,
không phải môi trường được tác giả cung cấp. Nếu đã import Transformers trước
khi cài, restart session/kernel rồi chạy lại từ đầu.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.51.3", "accelerate==1.6.0", "openpyxl==3.1.5",
                "huggingface_hub>=0.30,<1", "safetensors>=0.4.3"], check=True)


## 2. Kiểm tra GPU và cấu hình
Mặc định FP16, không lượng tử hóa. Nếu chỉ có GPU 16 GB, cell sẽ dừng trước
khi tải model. Có thể đổi `LOAD_IN_4BIT=True` và chạy lại cell này để thử bản NF4,
nhưng phải báo cáo đó là biến thể lượng tử hóa. Không đổi cấu hình giữa một run.
Các tham số greedy, seed, max_new_tokens và system message là thiết lập tái dựng.


In [ ]:
import os, json, hashlib, platform, random, time, uuid, csv, ast, re, shutil
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import importlib.metadata
import torch

MODEL_ID = "Qwen/Qwen2-7B-Instruct"
MODEL_REVISION = "f2826a00ceef68f0f2b946d945ecc0477ce4450c"  # Điền SHA từ run trước nếu muốn cố định cùng revision.
REPO_COMMIT = "2029d581d425dbe4f70136f08a06296fd774400f"
LOAD_IN_4BIT = False
SEED = 42
MAX_NEW_TOKENS = 256
MAX_INPUT_TOKENS = 8192  # Vượt ngưỡng sẽ báo lỗi, không âm thầm cắt event log.
SYSTEM_MESSAGE = "You are a helpful assistant."
ROOT = Path("/kaggle/working/llm4ias22_qwen2_rubric_v1")
ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / "author_artifacts"
ARTIFACTS.mkdir(exist_ok=True)
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Dùng HTTP download thông thường.

assert torch.cuda.is_available(), "Hãy bật GPU trong Notebook Settings; không dùng TPU cho notebook này."
GPU_INFO = [{"index": i, "name": torch.cuda.get_device_name(i),
             "total_GiB": round(torch.cuda.get_device_properties(i).total_memory / 2**30, 2),
             "free_GiB": round(torch.cuda.mem_get_info(i)[0] / 2**30, 2)}
            for i in range(torch.cuda.device_count())]
print(json.dumps(GPU_INFO, indent=2))
if not LOAD_IN_4BIT and sum(g["free_GiB"] for g in GPU_INFO) < 22:
    raise RuntimeError("FP16: nên có >=22 GiB GPU trống tổng cộng (ví dụ 2 T4). "
                       "Nếu thiếu, đổi LOAD_IN_4BIT=True, hoặc chọn phần cứng khác.")
if LOAD_IN_4BIT:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes>=0.45,<1"], check=True)
print("Mode:", "4-bit NF4; altered precision" if LOAD_IN_4BIT else "FP16; no quantization")

RUBRIC_TEXT = '# Rubric 2409 v1 — đã được người dùng chấp nhận, chưa phải rubric tác giả xác nhận\n\nNgày: 29/10/2026. Paper duy nhất làm gốc: **arXiv:2409.18009v2**.\n\nTrạng thái: người dùng chấp nhận v1 ngày 29/10/2026. Giữ nguyên các quy tắc F/C/R, U và cách báo cáo; mọi chỉnh sửa nội dung sau đây cần phiên bản mới. Sự chấp nhận này không có nghĩa tác giả đã xác nhận rubric.\n\n## 1. Phạm vi bài toán\n\nĐầu vào gồm mô tả agent/thiết bị, hàm điều khiển, SOP, nhiệm vụ và event log. Đầu ra là lý do cùng lệnh điều khiển tiếp theo. Tách SOP và Unexpected Events. Đây là điều khiển công nghiệp theo sự kiện, không phải dự đoán tuổi thọ thiết bị, QA chung hoặc lập lịch job-shop.\n\nMục V.B của [paper](https://arxiv.org/pdf/2409.18009v2) nêu hai chỉ số: Correctness Rate so command với reference và Reason Plausibility do con người chấm 1–5. Paper còn nói command có thể sai dù reason hợp lý. Giữ hai trục riêng, không ép điểm reason phải bằng độ đúng command.\n\nNhững gì dưới đây về tương đương ngữ nghĩa, mốc điểm và cách xử lý JSON là **quy tắc mình đề xuất để vận hành phép chấm**, không gán cho tác giả. Bản này được soạn sau khi đã xem kết quả 22 test, nên có tính exploratory; không tuyên bố đã preregister. Đóng băng phiên bản trước các lần chạy mới và áp dụng đối xứng cho hai nguồn output.\n\n## 2. Ba cột chấm độc lập\n\n### F — định dạng (0/1, tự động)\n\nF=1 khi toàn bộ response sau bỏ khoảng trắng đầu/cuối parse được thành JSON object, có reason và command kiểu string. Không bóc JSON từ văn xuôi, không sửa dấu nháy, không thực thi command. Kiểm tra cấu trúc lệnh/hàm hợp lệ riêng nếu cần; JSON hợp lệ chưa chứng minh điều khiển đúng.\n\nF=0 nếu sai điều kiện trên. Không sửa output gốc. Dấu code fence ngoài JSON vẫn là lỗi theo yêu cầu Only output a JSON structure của prompt; nếu thử parser linh hoạt phải báo cáo thành biến thể.\n\n### C — đúng câu lệnh theo reference (0/1/U, duyệt nội dung)\n\n- C=1: đúng hàm/hành động kế tiếp, đúng thiết bị và mọi tham số vận hành cần thiết theo reference cùng event cuối; không thêm hành động ngoài yêu cầu.\n- C=0: sai hàm, thiết bị, hướng, thời lượng, thứ tự; bỏ sót hành động cần; thêm hành động không được phép; hoặc xử lý sự cố khác policy tham chiếu.\n- C=U: reference mâu thuẫn với event/SOP, hoặc không xác định được command/ý định từ response. Phải ghi lý do, không tự biến U thành đúng hoặc bỏ khỏi bảng.\n\nQuy tắc tương đương:\n\n1. Bỏ khác biệt khoảng trắng, dấu nháy và cách biểu diễn số khi giá trị/nghĩa tương đương; không đổi hướng, thời lượng hoặc thiết bị.\n2. Với alert_to_supervisor(message), cho phép diễn đạt lại **cùng sự cố trên cùng thành phần**. Thông báo chung chung hoặc sai thiết bị không tự được chấp nhận.\n3. Nếu lỗi ngữ cảnh chỉ có một, tên thành phần có thể được lược nếu message vẫn định danh rõ sự cố qua ngữ cảnh; ghi chú quyết định để áp dụng cùng cách cho mọi output. Không dùng từ khóa alert đơn thuần làm đủ điều kiện.\n4. Không chấp nhận emergency_stop thay alert hoặc retry thay alert chỉ vì có vẻ hợp lý. Đó là hành động khác; nếu muốn policy rộng hơn phải định nghĩa trước như một phân tích riêng.\n5. Chuỗi nhiều lệnh chỉ đúng nếu toàn bộ chuỗi được specification cho phép; chứa lệnh đúng kèm lệnh sai không được tính đúng.\n6. Với F=0 nhưng người chấm vẫn đọc được command nguyên văn không mơ hồ, có thể chấm C về **ý định** và đánh dấu evidence_mode=verbatim_intent. Không chỉnh sửa hoặc gọi đó là output thực thi được. Nếu cần đo khả năng tiêu thụ tự động, dùng E ở dưới.\n\nChuẩn tham chiếu giữ nguyên. Nếu phát hiện lỗi nhãn, đặt reference_status=conflict, C=U cho cả hai output cùng test. Đề xuất sửa nhãn ở phiên bản khác kèm lý do; không sửa trực tiếp Excel.\n\n### R — Reason Plausibility (1–5/U, duyệt độc lập)\n\nĐánh giá reason dựa vào sự kiện/điều kiện/SOP, không dựa vào độ dài hoặc văn phong. Nếu reason đọc được trực tiếp trong response lỗi JSON, được đánh giá nội dung và ghi evidence_mode; F vẫn bằng 0. Không lấy reason từ cột ref_reason để thay reason mô hình.\n\n| R | Mốc chấm do mình đề xuất |\n|---|---|\n| 1 | Mâu thuẫn rõ với sự kiện, bịa trạng thái quyết định hoặc lập luận không liên quan |\n| 2 | Nhận ra một phần ngữ cảnh nhưng sai/thiếu sự kiện then chốt, lập luận khó hỗ trợ quyết định |\n| 3 | Có liên quan và không mâu thuẫn lớn, nhưng quá chung hoặc thiếu liên hệ với trạng thái mới nhất |\n| 4 | Bám đúng sự kiện quan trọng và giải thích hợp lý quyết định, còn thiếu chi tiết nhỏ |\n| 5 | Giải thích ngắn gọn, rõ nguyên nhân/điều kiện quyết định theo log và SOP/policy, không có giả định vô căn cứ |\n| U | Không có hoặc không xác định được reason; báo cáo thiếu riêng |\n\nReason đúng sự cố vẫn có thể R cao khi command chọn sai hàm. Không chấm reason=1 chỉ vì C=0. Ngược lại command khớp nhưng reason chung chung không tự nhận R=5.\n\n## 3. Chỉ số và mẫu số\n\nGiữ bảng all/SOP/Unexpected cho từng nguồn output, cùng tập test và cùng tiêu chí.\n\n- **AST match**: baseline kỹ thuật đã có, 4/22 mới và 5/22 tác giả. Giữ nguyên; không đổi tên nó thành semantic accuracy.\n- **Semantic command correctness**: C chấm như trên. Ghi số đúng k, sai w, chưa xác định u, N=k+w+u. Báo cáo khoảng [k/N, (k+u)/N], cùng tỷ lệ trên các test đã xác định k/(N-u) nếu N>u; phải in rõ coverage và mẫu số. Không trình bày một tỷ lệ đã loại U như thể đủ 22 test.\n- **Reason plausibility**: trung bình R trên các reason có điểm, kèm số reason có điểm/tổng và phân bố 1–5; không bỏ qua số missing/U. Không tự coi điểm H của tác giả là cùng mốc R này.\n- **Format validity**: tổng F/N.\n- **E=F×1[C=1]**: chỉ số bổ sung về tính hợp lệ định dạng + command đúng. C=U thì E chưa xác định nếu F=1, E=0 nếu F=0. Đây không phải metric paper công bố và không chứng minh thực thi đúng trên phần cứng.\n\nKhông gộp R vào accuracy, không dùng H=5 thay C, không so tỷ lệ 22 test với 65% trên 100 test. Chấm lại cả hai nguồn trước khi kết luận chênh lệch.\n\n## 4. Trình tự duyệt để hạn chế thiên lệch\n\n1. Đọc SOP, event và reference để lập case specification trước khi chấm output. Vì đã thấy output của run này, ghi nhận giới hạn đó.\n2. Đánh dấu case conflict dựa vào dữ liệu, áp dụng cho cả hai nguồn. Test 017 cần giữ U ở phép so dựa reference hiện tại.\n3. Dùng review_blind.jsonl; mỗi test có hai response mã A/B, không hiện model/source, điểm G/H hoặc kết quả AST. Sự khác biệt phong cách vẫn có thể làm người đọc đoán nguồn; không khẳng định blind tuyệt đối.\n4. Điền review_scores.csv: C, R, evidence_mode, ghi chú và quyết định case conflict. Không tự tính điểm semantic bằng regex hoặc LLM judge mà gọi là nhãn con người.\n5. Duyệt lại các ca không rõ, ghi rationale; một người chấm phải ghi là single-rater, không tuyên bố độ đồng thuận liên giám khảo.\n6. Chỉ mở answer_key.csv sau khi chấm xong. Tính các chỉ số cùng rubric/version. Nếu đổi quy tắc sau khi thấy nguồn, chấm lại cả hai bộ và tăng version.\n\n## 5. Những chỗ đang cần tác giả xác nhận\n\n- G, H và I trong workbook phản ánh các khái niệm nào; chỉ số nào dùng cho command accuracy.\n- Thông báo alert được chấp nhận theo ngữ nghĩa hay khớp chuỗi.\n- JSON lỗi có được tính đúng ý định và/hoặc đúng command hay không.\n- Policy cho emergency_stop/retry khi reference yêu cầu alert.\n- Reference của case 017 và chi tiết SOP C1/C2 ở BG56.\n\nKiểm tra toàn bộ ori_response của 22 dòng Qwen2 trong workbook: 008 là văn bản event log thay JSON; 014 và 016 có JSON lỗi dấu nháy. Vì vậy F của tác giả theo quy tắc strict đề xuất là 19/22. Đây là chỉ số mình tính lại từ workbook, không phải con số tác giả công bố. F của run mới đã kiểm tra trước đó là 22/22; không suy ra từ đó C hoặc R cao hơn.\n\nKhông gọi rubric này là nguyên bản tác giả. Tên phù hợp: **rubric tái dựng theo hai chỉ số của LLM4IAS, phiên bản v1**.\n'
RUBRIC_SHA256 = hashlib.sha256(RUBRIC_TEXT.encode()).hexdigest()


## 3. Tải đúng workbook và prompt của tác giả
Chỉ tải hai file ở commit cố định. Có kiểm tra SHA-256; không tự lấy bản mới
khi nguồn thay đổi. Không tải video hoặc clone repo nhiều lần.


In [ ]:
from urllib.request import urlopen, Request
EXPECTED_HASHES = {
    "evaluation_data.xlsx": "01e05db62a9ad86352d6fa553a3b1f2f309395262d11e39c9c6c20a687c37a9d",
    # Hash blob Git/HTTP gốc (LF), không dùng hash bản checkout Windows (CRLF).
    "prompt_example.txt": "867d80c496b55888f6a9bc2b3c943a230d05258b66fd764f3b863bcafab43be6",
}
for name, expected_hash in EXPECTED_HASHES.items():
    path = ARTIFACTS / name
    if not path.exists():
        url = f"https://raw.githubusercontent.com/YuchenXia/LLM4IAS/{REPO_COMMIT}/{name}"
        with urlopen(Request(url, headers={"User-Agent": "LLM4IAS-reproduction"}), timeout=90) as response:
            blob = response.read()
        assert hashlib.sha256(blob).hexdigest() == expected_hash, f"Hash mismatch: {name}"
        path.write_bytes(blob)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected_hash, f"Hash mismatch: {name}"
    print(name, path.stat().st_size, "bytes; SHA-256 OK")


## 4. Chuẩn bị đúng 22 test và raw tác giả
Chỉ dùng sheet Qwen2-7B-Instruct_1, 11 SOP + 11 Unexpected Events.
Case input/prompt không chứa đáp án hay output tác giả. Điểm G/H giữ làm
provenance nhưng không dùng thay nhãn C/R mới. Giữ raw kể cả JSON lỗi.


In [ ]:
import openpyxl
book = openpyxl.load_workbook(ARTIFACTS / "evaluation_data.xlsx", read_only=True, data_only=False)
try:
    rows = list(book["Qwen2-7B-Instruct_1"].iter_rows(values_only=True))
finally:
    book.close()
all_rows = [r for r in rows[1:] if isinstance(r[1], (int, float)) and isinstance(r[3], str)]
assert len(all_rows) == 100
selected = [r for r in all_rows if r[0] == "Inspection Station"]
assert len(selected) == 22
template = (ARTIFACTS / "prompt_example.txt").read_text(encoding="utf-8-sig")
prefix, _ = template.rsplit("\nInput:", 1)
CASES, REFERENCES = [], {}
for row in selected:
    case_id = f"inspection_station_{int(row[1]):03d}"
    prompt = prefix + "\nInput:\n" + row[3].strip() + "\n\nOutput:\n"
    CASES.append({"case_id":case_id, "type":row[2], "event_log":row[3].strip(), "prompt":prompt,
                  "prompt_sha256":hashlib.sha256(prompt.encode()).hexdigest()})
    assert isinstance(row[11], str) and row[11].strip(), "Missing author raw response"
    REFERENCES[case_id] = {"correct_command":row[4], "author_raw":row[11],
        "author_command":row[5], "author_judge_G":row[6], "author_rating_H":row[7],
        "reference_reason":row[10]}
assert len({c["case_id"] for c in CASES}) == 22
print("Prepared:",len(CASES),"cases",dict(Counter(c["type"] for c in CASES)))
print("Author source: Qwen2-7B-Instruct_1 / ori_response. References stay out of prompts.")
print("Conflict: inspection_station_017, C=U for both sources under accepted rubric.")


## 5. Tải tokenizer/model và lưu cấu hình
Tải checkpoint đầy đủ có thể mất thời gian và nhiều GB dung lượng mạng/đĩa.
Notebook cố định SHA checkpoint khi bắt đầu run; đây chưa chắc là SHA tác giả dùng.
FP16 chia model trên GPU với `device_map=auto`, batch size 1, không train.
Nếu lỗi hết bộ nhớ: restart session, chọn phần cứng khác hoặc chế độ 4-bit rõ ràng;
không retry chỉ vì model trả lời sai.


In [ ]:
from huggingface_hub import HfApi
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, set_seed
import transformers

MODEL_SHA = HfApi(token=False).model_info(MODEL_ID, revision=MODEL_REVISION).sha
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, token=False,
                                         trust_remote_code=False)
max_memory = {i: int(torch.cuda.mem_get_info(i)[0] * 0.78) for i in range(torch.cuda.device_count())}
load_kwargs = dict(revision=MODEL_SHA, token=False, trust_remote_code=False,
                   torch_dtype=torch.float16, device_map="auto", max_memory=max_memory,
                   low_cpu_mem_usage=True, attn_implementation="sdpa")
if LOAD_IN_4BIT:
    from transformers import BitsAndBytesConfig
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()
device_map = {str(k): str(v) for k, v in model.hf_device_map.items()}
print("Model SHA:", MODEL_SHA)
print("Device map:", device_map)
if any(v in ("cpu", "disk") for v in device_map.values()):
    raise RuntimeError("Model đang offload CPU/disk. Chọn GPU đủ bộ nhớ hoặc restart và dùng 4-bit.")
set_seed(SEED)
GENERATION = GenerationConfig(do_sample=False, num_beams=1, max_new_tokens=MAX_NEW_TOKENS,
    use_cache=True, eos_token_id=model.generation_config.eos_token_id,
    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id)
RUN_DIR = ROOT / "runs" / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6])
(RUN_DIR / "responses").mkdir(parents=True)
(RUN_DIR / "prompts").mkdir()
packages = {name: importlib.metadata.version(name) for name in
            ["torch", "transformers", "accelerate", "openpyxl", "huggingface_hub", "safetensors"]}
if LOAD_IN_4BIT:
    packages["bitsandbytes"] = importlib.metadata.version("bitsandbytes")
METADATA = {"model_id": MODEL_ID, "model_sha": MODEL_SHA, "repo_commit": REPO_COMMIT,
    "paper": "arXiv:2409.18009v2", "load_in_4bit": LOAD_IN_4BIT, "dtype": "float16",
    "seed": SEED, "system_message": SYSTEM_MESSAGE, "generation": GENERATION.to_dict(),
    "max_input_tokens": MAX_INPUT_TOKENS, "gpu": GPU_INFO, "device_map": device_map,
    "packages": packages, "python": platform.python_version(), "cuda": torch.version.cuda,
    "author_file_hashes": EXPECTED_HASHES,
    "scope": "22 Inspection Station cases; reconstructed prompts/settings; not author source code",
    "rubric_version": "2409_v1_user_accepted_2026-10-29", "rubric_sha256": RUBRIC_SHA256,
    "author_sheet": "Qwen2-7B-Instruct_1", "C_R_scoring": "manual review; auto suggestions are provisional"}
(RUN_DIR / "metadata.json").write_text(json.dumps(METADATA, ensure_ascii=False, indent=2), encoding="utf-8")
(RUN_DIR / "pip_freeze.txt").write_text(subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")
for case in CASES:
    (RUN_DIR / "prompts" / (case["case_id"] + ".txt")).write_text(case["prompt"], encoding="utf-8")
print("Run directory:", RUN_DIR)

(RUN_DIR / "rubric_2409_v1.md").write_text(RUBRIC_TEXT, encoding="utf-8")


## 6. Hàm sinh một lần và chấm câu lệnh
Không thực thi command. Chuẩn hóa bằng AST để bỏ khác biệt dấu nháy/khoảng trắng;
không tự coi các chuỗi nhiều hành động là tương đương. Đây là metric kỹ thuật
do mình định nghĩa, không thay thế rubric của tác giả hoặc đánh giá thực thi vật lý.
JSON không hợp lệ tính là thất bại định dạng, giữ nguyên output để đọc thủ công.


In [ ]:
def canonical_command(command):
    if not isinstance(command, str) or not command.strip():
        return None
    try:
        node = ast.parse(command.strip(), mode="eval").body
    except (SyntaxError, ValueError, TypeError):
        return None
    if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
        return None
    return ast.dump(node, annotate_fields=True, include_attributes=False)


def parse_output(raw):
    try:
        obj = json.loads(raw.strip())
    except (ValueError, TypeError):
        return {"json_valid": False, "schema_valid": False, "command": None, "reason": None}
    is_dict = isinstance(obj, dict)
    valid = is_dict and isinstance(obj.get("command"), str) and isinstance(obj.get("reason"), str)
    return {"json_valid": True, "schema_valid": valid,
            "command": obj.get("command") if is_dict else None,
            "reason": obj.get("reason") if is_dict else None}


def synchronize_gpus():
    for i in range(torch.cuda.device_count()):
        torch.cuda.synchronize(i)


def infer_case(case):
    result_path = RUN_DIR / "responses" / (case["case_id"] + ".json")
    if result_path.exists():
        existing = json.loads(result_path.read_text(encoding="utf-8"))
        assert existing["prompt_sha256"] == case["prompt_sha256"]
        if existing.get("error"):
            raise RuntimeError("Đã có lỗi ở test này; xem file lỗi và tạo run mới sau khi sửa môi trường.")
        return existing  # Không chạy lại test đầu sau smoke test.
    messages = [{"role": "system", "content": SYSTEM_MESSAGE},
                {"role": "user", "content": case["prompt"]}]
    rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    batch = tokenizer(rendered, return_tensors="pt", add_special_tokens=False)
    input_tokens = batch["input_ids"].shape[-1]
    record = {"case_id": case["case_id"], "type": case["type"],
              "prompt_sha256": case["prompt_sha256"],
              "rendered_prompt_sha256": hashlib.sha256(rendered.encode()).hexdigest(),
              "input_tokens": input_tokens}
    try:
        if input_tokens > MAX_INPUT_TOKENS:
            raise ValueError("Prompt vượt MAX_INPUT_TOKENS; không cắt dữ liệu tự động.")
        batch = batch.to(model.get_input_embeddings().weight.device)
        set_seed(SEED)
        synchronize_gpus()
        start = time.perf_counter()
        with torch.inference_mode():
            generated = model.generate(**batch, generation_config=GENERATION)
        synchronize_gpus()
        record["latency_seconds"] = time.perf_counter() - start
        output_ids = generated[0, input_tokens:]
        raw = tokenizer.decode(output_ids, skip_special_tokens=True)
        record.update(raw_output=raw, output_tokens=int(output_ids.numel()),
                      hit_generation_limit=bool(output_ids.numel() >= MAX_NEW_TOKENS),
                      error=None, **parse_output(raw))
    except Exception as exc:
        record.update(error=type(exc).__name__ + ": " + str(exc))
        result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
        raise
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    return record


"""F/C/R review helpers. No model calls, no execution/repair of generated commands."""
import ast
import csv
from decimal import Decimal
import hashlib
import json
from pathlib import Path
import random

RUBRIC_VERSION = "2409_v1_user_accepted_2026-10-29"
CONFLICT_CASES = {"inspection_station_017"}
# Public prompt function signatures; used only by evaluator, never by inference.
ARGUMENT_NAMES = {
    "conveyor_1_run": ("direction", "time"), "conveyor_2_run": ("direction", "time"),
    "conveyor_3_run": ("direction", "time"), "conveyor_4_run": ("direction", "time"),
    "switch_actuate": ("action",), "alert_to_supervisor": ("reason",),
    "RFID_read_workpiece_info": (), "transport_robot_request": (),
    "release_holder_H1": (), "release_holder_H2": (), "release_holder_H3": (),
    "emergency_stop": (),
}


def rubric_parse(raw):
    try:
        obj = json.loads(raw.strip())
    except (ValueError, TypeError, AttributeError):
        return {"F": 0, "command": None, "reason": None}
    if not isinstance(obj, dict):
        return {"F": 0, "command": None, "reason": None}
    good = all(isinstance(obj.get(k), str) for k in ("reason", "command"))
    return {"F": int(good), "command": obj.get("command"), "reason": obj.get("reason")}


def strict_ast(command):
    try:
        node = ast.parse(command.strip(), mode="eval").body
    except (SyntaxError, ValueError, TypeError, AttributeError):
        return None
    return ast.dump(node, annotate_fields=True, include_attributes=False) if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) else None


def normalized_literal(value):
    if isinstance(value, bool):
        return ("bool", value)
    if isinstance(value, (int, float)):
        return ("number", Decimal(str(value)))
    if isinstance(value, str) or value is None:
        return (type(value).__name__, value)
    if isinstance(value, (tuple, list)):
        return (type(value).__name__, tuple(normalized_literal(v) for v in value))
    raise ValueError("Unsupported argument literal")


def command_signature(command):
    try:
        node = ast.parse(command.strip(), mode="eval").body
        if not isinstance(node, ast.Call) or not isinstance(node.func, ast.Name):
            return None
        if any(k.arg is None for k in node.keywords) or len({k.arg for k in node.keywords}) != len(node.keywords):
            return None
        args = [normalized_literal(ast.literal_eval(a)) for a in node.args]
        kwargs = {k.arg: normalized_literal(ast.literal_eval(k.value)) for k in node.keywords}
        if node.func.id in ARGUMENT_NAMES:
            params = ARGUMENT_NAMES[node.func.id]
            if len(args) > len(params) or any(k not in params for k in kwargs):
                return None
            bound = dict(zip(params,args))
            if set(bound) & set(kwargs):
                return None
            bound.update(kwargs)
            if set(bound) != set(params):
                return None
            return (node.func.id, tuple(bound[p] for p in params), ())
        return (node.func.id, tuple(args), tuple(sorted(kwargs.items())))
    except (ValueError, SyntaxError, TypeError, AttributeError):
        return None


def suggest_C(case_id, raw, reference):
    parsed = rubric_parse(raw)
    if case_id in CONFLICT_CASES:
        return "U", "Reference conflicts with event/ref_reason; keep U for both sources."
    if not parsed["F"]:
        return "U", "Invalid format: inspect original text for verbatim intent; no repair."
    predicted, target = command_signature(parsed["command"]), command_signature(reference)
    if target is None:
        return "U", "Reference cannot be parsed by this helper."
    if predicted == target:
        return "1", "Single reference call matches after literal normalization; reviewer must confirm."
    if predicted and predicted[0] == target[0] == "alert_to_supervisor":
        return "U", "Review alert message semantics against device/fault; never score by tool name alone."
    return "0", "Different action/operational arguments or unsupported/multiple call; reviewer must confirm."


def export_review(cases, results, references, run_dir):
    """Create 44 masked items and suggestions separately. Preserve manual CSV on rerun."""
    run_dir = Path(run_dir)
    assert len(cases) == len(results) == 22
    new = {r["case_id"]: r for r in results}
    assert len(new) == 22 and set(new) == {c["case_id"] for c in cases}
    rng = random.Random(2409)
    items, key, manual, auto = [], [], [], []
    md = ["# Review 22 test / 44 responses\n\nRead rubric_2409_v1.md first. "
          "A/B source assignments vary per case. Do not open source_key.csv before review. "
          "Source masking cannot guarantee the reviewer cannot guess from style.\n"]
    for case in cases:
        cid = case["case_id"]
        ref = references[cid]
        assert new[cid].get("raw_output") is not None and not new[cid].get("error")
        assert new[cid]["prompt_sha256"] == case["prompt_sha256"]
        pair = [("new_run", new[cid]["raw_output"]), ("author_saved_run", ref["author_raw"])]
        assert all(isinstance(raw, str) for _,raw in pair)
        rng.shuffle(pair)
        md.append(f"\n## {cid} — {case['type']}\n\nReference: `{ref['correct_command']}`\n\n"
                  + ("REFERENCE CONFLICT: keep C=U.\n\n" if cid in CONFLICT_CASES else "")
                  + "Event log:\n\n```text\n" + case["event_log"] + "\n```\n")
        for label, (source, raw) in zip(("A", "B"), pair):
            item_id = cid + "_" + label
            parsed = rubric_parse(raw)
            suggestion, why = suggest_C(cid, raw, ref["correct_command"])
            strict = strict_ast(parsed["command"])
            matched = bool(parsed["F"] and strict is not None and strict == strict_ast(ref["correct_command"]))
            items.append({"item_id": item_id, "case_id": cid, "type": case["type"],
                          "event_log": case["event_log"], "reference_command": ref["correct_command"],
                          "reference_reason": ref["reference_reason"], "response_raw": raw,
                          "F_auto": parsed["F"], "reference_conflict": cid in CONFLICT_CASES})
            key.append({"item_id": item_id, "source": source})
            manual.append({"item_id": item_id, "C_final_0_1_U": "", "R_final_1_to_5_U": "",
                           "evidence_mode": "", "reviewer": "", "review_notes": ""})
            auto.append({"item_id": item_id, "case_id": cid, "type": case["type"], "source": source,
                         "F": parsed["F"], "strict_ast_match": int(matched),
                         "C_suggested": suggestion, "suggestion_note": why,
                         "command": parsed["command"], "reason": parsed["reason"]})
            md.append(f"\n### Response {label}\n\n```text\n{raw}\n```\n")
    (run_dir / "review_blind.jsonl").write_text("".join(json.dumps(x, ensure_ascii=False)+"\n" for x in items), encoding="utf-8")
    (run_dir / "review_blind.md").write_text("".join(md), encoding="utf-8")
    for name, rows in (("source_key.csv",key),("auto_suggestions.csv",auto),("review_scores.csv",manual)):
        p = run_dir / name
        if name == "review_scores.csv" and p.exists():
            continue
        with p.open("w", newline="", encoding="utf-8-sig") as file:
            w = csv.DictWriter(file, fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
    context = {"rubric_version": RUBRIC_VERSION, "items": len(items), "case_count": len(cases),
               "state": "C suggestions are provisional; no R labels generated; manual scores pending"}
    (run_dir / "review_manifest.json").write_text(json.dumps(context, ensure_ascii=False, indent=2), encoding="utf-8")
    return items, auto


def summarize_review(items, auto, score_path):
    with Path(score_path).open(encoding="utf-8-sig", newline="") as file:
        manual_rows = list(csv.DictReader(file))
    manual = {r["item_id"]:r for r in manual_rows}
    expected_ids = {i["item_id"] for i in items}
    assert len(manual_rows) == len(manual) == len(items) and set(manual) == expected_ids, "Missing/duplicate/unknown review IDs"
    auto_map = {a["item_id"]: a for a in auto}
    source = {a["item_id"]: a["source"] for a in auto}
    checked = []
    for item in items:
        row = manual[item["item_id"]]
        c = row["C_final_0_1_U"].strip().upper()
        r = row["R_final_1_to_5_U"].strip().upper()
        if c not in ("", "0", "1", "U") or r not in ("", "1", "2", "3", "4", "5", "U"):
            raise ValueError("Invalid manual score: " + item["item_id"])
        if item["reference_conflict"] and c not in ("", "U"):
            raise ValueError("Reference conflict is U for both sources under rubric v1: " + item["item_id"])
        if not item["F_auto"] and (c in ("0", "1") or r in ("1", "2", "3", "4", "5")) and row["evidence_mode"] != "verbatim_intent":
            raise ValueError("Invalid-format content needs evidence_mode=verbatim_intent: " + item["item_id"])
        if (c and c != auto_map[item["item_id"]]["C_suggested"]) and not row["review_notes"].strip():
            raise ValueError("Document why manual C differs from suggestion: " + item["item_id"])
        checked.append({**item,"source":source[item["item_id"]],"C_final":c,"R_final":r})
    summary = {"rubric_version":RUBRIC_VERSION,"manual_scores_complete":all(x["C_final"] and x["R_final"] for x in checked),"sources":{}}
    for label in ("new_run", "author_saved_run"):
        summary["sources"][label] = {}
        for group in ("all", "SOP", "Unexpected Events"):
            subset=[x for x in checked if x["source"]==label and (group=="all" or x["type"]==group)]
            n=len(subset);k=sum(x["C_final"]=="1" for x in subset);w=sum(x["C_final"]=="0" for x in subset)
            u=sum(x["C_final"]=="U" for x in subset);pending=sum(not x["C_final"] for x in subset)
            reasons=[int(x["R_final"]) for x in subset if x["R_final"] in ("1","2","3","4","5")]
            e_correct=sum(x["F_auto"]==1 and x["C_final"]=="1" for x in subset)
            e_unknown=sum(x["F_auto"]==1 and x["C_final"] in ("","U") for x in subset)
            summary["sources"][label][group]={"n":n,"F_valid":sum(x["F_auto"] for x in subset),
                "strict_ast_matches":sum(auto_map[x["item_id"]]["strict_ast_match"] for x in subset),
                "C_manual_correct":k,"C_manual_wrong":w,"C_manual_U":u,"C_pending":pending,
                "C_accuracy_bounds":[k/n,(k+u+pending)/n],"C_determined_coverage":(k+w)/n,
                "C_accuracy_on_determined":k/(k+w) if k+w else None,
                "R_mean":sum(reasons)/len(reasons) if reasons else None,"R_scored_count":len(reasons),
                "R_distribution":{str(value):reasons.count(value) for value in range(1,6)},
                "R_U":sum(x["R_final"]=="U" for x in subset),"R_pending":sum(not x["R_final"] for x in subset),
                "E_correct":e_correct,"E_unknown_or_pending":e_unknown,
                "C_suggestions_only":{s:sum(auto_map[x["item_id"]]["C_suggested"]==s for x in subset) for s in ("0","1","U")}}
    return summary


## 7. Smoke test một trường hợp
Một output sai không có nghĩa code lỗi. Không sửa prompt/seed cho đến khi đúng.
Nếu model sinh được phản hồi thì tiếp tục chạy tất cả, giữ cả những câu trả lời sai.


In [ ]:
first = infer_case(CASES[0])
print(first["raw_output"])
print("Reference (chỉ dùng sau inference):", REFERENCES[CASES[0]["case_id"]]["correct_command"])
print("Tokens:", first["input_tokens"], first["output_tokens"], "Latency:", first["latency_seconds"])


## 8. Chạy đủ 22 trường hợp
Chạy tuần tự, không batch nhiều test; lưu output sau từng test.
Test smoke ở trên được tái sử dụng, không gọi lại để chọn đáp án đẹp hơn.


In [ ]:
RESULTS = []
for index, case in enumerate(CASES, 1):
    result = infer_case(case)
    RESULTS.append(result)
    print(f"{index:02d}/{len(CASES)} {case['case_id']} JSON={result['schema_valid']} "
          f"tokens={result['output_tokens']} time={result['latency_seconds']:.1f}s")
assert len(RESULTS) == 22


## 9. Xuất bộ review theo rubric mới và chấm tự động phần xác định được
C_suggested=1 cho single call khớp reference sau chuẩn hóa literal;
C_suggested=0 cho sai hành động/tham số rõ ràng; U cho alert paraphrase,
raw lỗi định dạng hoặc reference conflict. Đây là gợi ý, chưa phải C_final.
Case 017 phải U cho hai nguồn; không đổi nhãn để nâng điểm.
R để trống đến khi chấm thủ công. File review_scores.csv được giữ khi chạy
lại cell, không ghi đè nhãn bạn đã điền. Không mở source_key.csv hoặc
auto_suggestions.csv trước khi blind review nếu muốn giảm thiên lệch.


In [ ]:
REVIEW_ITEMS, AUTO_ROWS = export_review(CASES, RESULTS, REFERENCES, RUN_DIR)
# Manual file starts blank. This summary explicitly shows pending, not fake accuracy.
RUBRIC_SUMMARY = summarize_review(REVIEW_ITEMS, AUTO_ROWS, RUN_DIR / "review_scores.csv")
(RUN_DIR / "rubric_summary.json").write_text(json.dumps(RUBRIC_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")
for source, groups in RUBRIC_SUMMARY["sources"].items():
    for group, s in groups.items():
        print(source, group, f"n={s['n']} F={s['F_valid']}/{s['n']} AST={s['strict_ast_matches']}/{s['n']}",
              "C suggestions (not final):",s["C_suggestions_only"],
              "C pending:",s["C_pending"],"R pending:",s["R_pending"])
print("Review 44 responses in review_blind.md; enter labels in review_scores.csv.")
print("No semantic accuracy or reason score has been finalized by this cell.")


## 10. Tổng hợp nhãn thủ công (có thể để sau khi tải ZIP)
Mở review_blind.md, đọc rubric, điền CSV: C_final_0_1_U, R_final_1_to_5_U,
evidence_mode, reviewer và notes. Với raw JSON lỗi mà vẫn chấm được nội dung,
dùng evidence_mode=verbatim_intent. U không đồng nghĩa sai; không loại khỏi
mẫu số mà không báo coverage. Không tự gán R=5 chỉ vì command đúng.
Nếu chưa điền, có thể chạy cell để thấy pending hoặc đi thẳng đến ZIP.
Nếu upload CSV đã chấm thành Kaggle input, sửa REVIEW_CSV_PATH cho đúng file.


In [ ]:
REVIEW_CSV_PATH = RUN_DIR / "review_scores.csv"
RUBRIC_SUMMARY = summarize_review(REVIEW_ITEMS, AUTO_ROWS, REVIEW_CSV_PATH)
if Path(REVIEW_CSV_PATH).resolve() != (RUN_DIR / "review_scores.csv").resolve():
    shutil.copyfile(REVIEW_CSV_PATH, RUN_DIR / "review_scores.csv")
(RUN_DIR / "rubric_summary.json").write_text(json.dumps(RUBRIC_SUMMARY, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(RUBRIC_SUMMARY, ensure_ascii=False, indent=2))
if not RUBRIC_SUMMARY["manual_scores_complete"]:
    print("Manual C/R not complete. Pending/U are explicit; do not report a finalized semantic accuracy.")


## 11. ZIP kết quả và tải về
ZIP gồm raw mới, raw tác giả trong review pack, rubric snapshot, metadata,
gợi ý tự động, review_scores.csv và rubric_summary.json. Không chứa trọng số.
Có thể ZIP khi C/R còn pending; trạng thái chưa chấm phải được giữ rõ.


In [ ]:
zip_path = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Download:", zip_path)
from IPython.display import FileLink, display
display(FileLink(zip_path))
